# 15 · Checking the rating-method choice

The chosen method (pressure-weighted) against the best alternative for each side, on three independent checks:
the 52 hidden-match tests, the 2026 season, and the outside matches.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes</summary>

P14b - Evidence for the rating-model family decision (2026-11-12 review), on ONE yardstick, three ways:  
 (i) P9's 52 unseen-match splits, (ii) the 2026 season, (iii) the public-data exam (duplicates removed in P14).  
CHOSEN = the pressure-weighted settings in p09_summary; ALTERNATIVE = what P9 picked when every model family was allowed  
(batters: R2-joint, bowlers: R3). Players not available for selection are left out of the exam pools, as in P14.
</details>

In [2]:
import os
import numpy as np
import pandas as pd

%run ./11a_settings_test_setup.ipynb
log = get_logger("p14b_family_check")
from rrlib.common import load, load_json, save_table, weighted_spearman, excluded_players

EXCL = excluded_players()
_ch = load_json("p09_summary")["chosen"]
CFGS = {
    ("bat", "ALTERNATIVE: all balls (R2-joint)"): {"value": "v3", "model": "R2-joint", "w_d": 0.0, "w_c": 0.0, "k_mult": 2.0, "min_n": _ch["bat"].get("min_n", 0)},
    ("bat", "CHOSEN: pressure-weighted"): _ch["bat"],
    ("bowl", "ALTERNATIVE: pressure balls only (R3)"): {"value": "v1", "model": "R3", "w_d": 0.0, "w_c": 0.0, "k_mult": 2.0, "min_n": _ch["bowl"].get("min_n", 0)},
    # every option uses the same evidence floor as the chosen one, so the comparison is like for like
    ("bowl", "CHOSEN: pressure-weighted"): _ch["bowl"],
}
# the stage-by-stage rule's own pick within the family, where a review decision changed it
_rp = load_json("p09_summary").get("rule_pick", {})
for _s in ("bat", "bowl"):
    if _s in _rp and {k: v for k, v in _rp[_s].items() if k != "min_n"} != {k: v for k, v in _ch[_s].items() if k != "min_n"}:
        CFGS[(_s, f"RULE_PICK: pressure-weighted (a{_rp[_s].get('alpha')}, k{_rp[_s].get('k_mult')})")] = dict(_rp[_s], min_n=_ch[_s].get("min_n", 0))
rows, per = [], {}
# (i) 52 unseen-match splits
for (side, nm), cfg in CFGS.items():
    res = evaluate(cfg, (side,))
    per[(side, nm)] = res
    r = res[res.side == side]
    rows.append({"side": side, "setting": nm, "config": str(cfg), "test": "(i) 52 unseen-match splits",
                 "rank_corr": 0.5 * r[r.kind == "random"].rho_v2.mean() + 0.5 * r[r.kind == "rolling"].rho_v2.mean(),
                 "rank_corr_random50": r[r.kind == "random"].rho_v2.mean(), "rank_corr_rolling2": r[r.kind == "rolling"].rho_v2.mean(),
                 "top10_lift_winprob_pts_per_ball": 100 * r.top10_lift_v2.mean()})
    log.info(str(rows[-1]))
for side in ("bat", "bowl"):
    kc = [k for k in per if k[0] == side and k[1].startswith("CHOSEN")][0]
    a = per[kc].query("kind == 'random'").set_index("split").rho_v2
    for other in ("ALTERNATIVE", "RULE_PICK"):
        ko_ = [k for k in per if k[0] == side and k[1].startswith(other)]
        if not ko_:
            continue
        ko = ko_[0]
        dd = a - per[ko].query("kind == 'random'").set_index("split").rho_v2.reindex(a.index)
        rows.append({"side": side, "setting": f"CHOSEN minus {other}", "test": "(i) paired, 50 random splits",
                     "rank_corr": dd.mean(), "paired_se": dd.std() / np.sqrt(len(dd)), "chosen_better_share": (dd > 0).mean()})
        log.info(str(rows[-1]))

19:01:20 52 splits


19:01:34 {'side': 'bat', 'setting': 'ALTERNATIVE: all balls (R2-joint)', 'config': "{'value': 'v3', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.14124936608860417), 'rank_corr_random50': np.float64(0.14646359285610103), 'rank_corr_rolling2': np.float64(0.13603513932110733), 'top10_lift_winprob_pts_per_ball': np.float64(0.2767044622015222)}


19:01:43 {'side': 'bat', 'setting': 'CHOSEN: pressure-weighted', 'config': "{'value': 'v3', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.12648402765461547), 'rank_corr_random50': np.float64(0.14364942174218429), 'rank_corr_rolling2': np.float64(0.10931863356704663), 'top10_lift_winprob_pts_per_ball': np.float64(0.24600436389000246)}


19:01:51 {'side': 'bowl', 'setting': 'ALTERNATIVE: pressure balls only (R3)', 'config': "{'value': 'v1', 'model': 'R3', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.0975811281550773), 'rank_corr_random50': np.float64(0.12364306255555568), 'rank_corr_rolling2': np.float64(0.07151919375459892), 'top10_lift_winprob_pts_per_ball': np.float64(0.21537570450871849)}


19:01:58 {'side': 'bowl', 'setting': 'CHOSEN: pressure-weighted', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.09871540805482734), 'rank_corr_random50': np.float64(0.10931418036034722), 'rank_corr_rolling2': np.float64(0.08811663574930748), 'top10_lift_winprob_pts_per_ball': np.float64(0.18802362262054353)}


19:02:06 {'side': 'bat', 'setting': 'RULE_PICK: pressure-weighted (a0.5, k2.0)', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.11894816157206861), 'rank_corr_random50': np.float64(0.1358755769812292), 'rank_corr_rolling2': np.float64(0.102020746162908), 'top10_lift_winprob_pts_per_ball': np.float64(0.2390483370432257)}


19:02:14 {'side': 'bowl', 'setting': 'RULE_PICK: pressure-weighted (a0.5, k0.5)', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 60, 'alpha': 0.5}", 'test': '(i) 52 unseen-match splits', 'rank_corr': np.float64(0.09923072336304198), 'rank_corr_random50': np.float64(0.10908949910966688), 'rank_corr_rolling2': np.float64(0.08937194761641706), 'top10_lift_winprob_pts_per_ball': np.float64(0.1861088581730221)}


19:02:14 {'side': 'bat', 'setting': 'CHOSEN minus ALTERNATIVE', 'test': '(i) paired, 50 random splits', 'rank_corr': np.float64(-0.0028141711139167552), 'paired_se': np.float64(0.0016262259485425251), 'chosen_better_share': np.float64(0.46)}


19:02:14 {'side': 'bat', 'setting': 'CHOSEN minus RULE_PICK', 'test': '(i) paired, 50 random splits', 'rank_corr': np.float64(0.00777384476095507), 'paired_se': np.float64(0.0026235195552758183), 'chosen_better_share': np.float64(0.66)}


19:02:14 {'side': 'bowl', 'setting': 'CHOSEN minus ALTERNATIVE', 'test': '(i) paired, 50 random splits', 'rank_corr': np.float64(-0.014328882195208462), 'paired_se': np.float64(0.004576797016294831), 'chosen_better_share': np.float64(0.34)}


19:02:14 {'side': 'bowl', 'setting': 'CHOSEN minus RULE_PICK', 'test': '(i) paired, 50 random splits', 'rank_corr': np.float64(0.00022468125068033512), 'paired_se': np.float64(0.0002040286303287798), 'chosen_better_share': np.float64(0.56)}


#### (ii) the 2026 season (fit on 2022-25)

In [3]:
train, test = V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]
for (side, nm), cfg in CFGS.items():
    y = yardstick(test, side)
    sc = RL.build_scores(train, side, cfg, comp_info).score
    tr_ids = set(train.loc[train.is_pressure & train[f"v1_{side}"].notna(), RL.PCOL[side]].unique())
    pool = y.index[y.index.isin(tr_ids)]
    s, yy = sc.reindex(pool).fillna(-1e9), y.loc[pool]
    top = s.sort_values(ascending=False).index[:10]
    rs = np.random.RandomState(SEED); bs = []
    for _ in range(500):
        ix = rs.randint(0, len(pool), len(pool))
        bs.append(weighted_spearman(s.values[ix], yy.y2.values[ix], yy.n.values[ix]))
    rows.append({"side": side, "setting": nm, "config": str(cfg), "test": "(ii) 2026 season", "players": len(pool),
                 "rank_corr": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                 "rank_corr_lo90": np.quantile(bs, 0.05), "rank_corr_hi90": np.quantile(bs, 0.95),
                 "top10_lift_winprob_pts_per_ball": 100 * (np.average(yy.loc[top, "y2"], weights=yy.loc[top, "n"]) - np.average(yy.y2, weights=yy.n))})
    log.info(str(rows[-1]))

19:02:14 {'side': 'bat', 'setting': 'ALTERNATIVE: all balls (R2-joint)', 'config': "{'value': 'v3', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60}", 'test': '(ii) 2026 season', 'players': 77, 'rank_corr': -0.05754097059020998, 'rank_corr_lo90': np.float64(-0.25016043374915714), 'rank_corr_hi90': np.float64(0.12965754062023868), 'top10_lift_winprob_pts_per_ball': np.float64(0.039240527649692165)}


19:02:15 {'side': 'bat', 'setting': 'CHOSEN: pressure-weighted', 'config': "{'value': 'v3', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(ii) 2026 season', 'players': 77, 'rank_corr': -0.05196605139484407, 'rank_corr_lo90': np.float64(-0.25392058507093346), 'rank_corr_hi90': np.float64(0.12781295808204948), 'top10_lift_winprob_pts_per_ball': np.float64(-0.22879162822632668)}


19:02:15 {'side': 'bowl', 'setting': 'ALTERNATIVE: pressure balls only (R3)', 'config': "{'value': 'v1', 'model': 'R3', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60}", 'test': '(ii) 2026 season', 'players': 89, 'rank_corr': 0.21319247307475692, 'rank_corr_lo90': np.float64(0.03189306116497694), 'rank_corr_hi90': np.float64(0.38708600197879367), 'top10_lift_winprob_pts_per_ball': np.float64(0.24899637684263667)}


19:02:16 {'side': 'bowl', 'setting': 'CHOSEN: pressure-weighted', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(ii) 2026 season', 'players': 89, 'rank_corr': 0.1717557100838039, 'rank_corr_lo90': np.float64(-0.014910602247158887), 'rank_corr_hi90': np.float64(0.35073507265688303), 'top10_lift_winprob_pts_per_ball': np.float64(0.5359562858063024)}


19:02:16 {'side': 'bat', 'setting': 'RULE_PICK: pressure-weighted (a0.5, k2.0)', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}", 'test': '(ii) 2026 season', 'players': 77, 'rank_corr': -0.07653364175971869, 'rank_corr_lo90': np.float64(-0.2650984390020639), 'rank_corr_hi90': np.float64(0.09605633451004629), 'top10_lift_winprob_pts_per_ball': np.float64(-0.1740227723113304)}


19:02:16 {'side': 'bowl', 'setting': 'RULE_PICK: pressure-weighted (a0.5, k0.5)', 'config': "{'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 60, 'alpha': 0.5}", 'test': '(ii) 2026 season', 'players': 89, 'rank_corr': 0.17454731725419811, 'rank_corr_lo90': np.float64(-0.008844404742637505), 'rank_corr_hi90': np.float64(0.3480730058786767), 'top10_lift_winprob_pts_per_ball': np.float64(0.5359562858063024)}


#### (iii) public-data exam: same pools and yardstick as P14

In [4]:
vx = load("values_ext")
players = load("players").set_index("player_id")
G = {"all public matches": np.ones(len(vx), bool), "T20 World Cups": vx.is_wc.eq(1).values,
     "franchise leagues": vx.level.isin(["BBL", "PSL", "CPL", "SA20", "ILT20", "MLC", "BPL", "LPL", "Blast"]).values,
     "full-member T20Is": vx.level.eq("T20I-full").values}
full_scores = {k: RL.build_scores(V_ALL, k[0], cfg, comp_info).score for k, cfg in CFGS.items()}
for gname, gm in G.items():
    for (side, nm), sc in full_scores.items():
        pc = RL.PCOL[side]
        d = vx[gm & vx.is_pressure & vx[f"wpa_{side}"].notna()]
        y = d.groupby(pc).agg(y=(f"wpa_{side}", "mean"), n=(f"wpa_{side}", "size"))
        base_ids = set(V_ALL.loc[V_ALL.is_pressure & V_ALL[f"v1_{side}"].notna(), pc].unique())
        y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
        ss = sc.reindex(y.index).fillna(-1e9)
        top = ss.sort_values(ascending=False).index[:10]
        rows.append({"side": side, "setting": nm, "config": str(CFGS[(side, nm)]), "test": f"(iii) public: {gname}", "players": len(y),
                     "rank_corr": weighted_spearman(ss.values, y.y.values, y.n.values),
                     "top10_lift_winprob_pts_per_ball": 100 * (np.average(y.loc[top, "y"], weights=y.loc[top, "n"]) - np.average(y.y, weights=y.n))})
R = pd.DataFrame(rows)
save_table(R, "rating_family_check")
log.info("Rating family check:\n" + R.drop(columns="config").round(4).to_string(index=False))

19:02:19 Rating family check:
side                                   setting                             test  rank_corr  rank_corr_random50  rank_corr_rolling2  top10_lift_winprob_pts_per_ball  paired_se  chosen_better_share  players  rank_corr_lo90  rank_corr_hi90
 bat         ALTERNATIVE: all balls (R2-joint)       (i) 52 unseen-match splits     0.1412              0.1465              0.1360                           0.2767        NaN                  NaN      NaN             NaN             NaN
 bat                 CHOSEN: pressure-weighted       (i) 52 unseen-match splits     0.1265              0.1436              0.1093                           0.2460        NaN                  NaN      NaN             NaN             NaN
bowl     ALTERNATIVE: pressure balls only (R3)       (i) 52 unseen-match splits     0.0976              0.1236              0.0715                           0.2154        NaN                  NaN      NaN             NaN             NaN
bowl                 C